In [18]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

X_train = pd.read_csv("data/X_train_purged_df.csv", sep=',')
mapping = {"Absence" : 0, "Presence" : 1}
X_train["Heart Disease"] = X_train["Heart Disease"].map(mapping)
y = X_train["Heart Disease"]
X_train.drop(columns="Heart Disease", inplace=True)

X_test = pd.read_csv("data/X_test_kaggle_purged_df.csv", sep=',')
df_test = pd.read_csv("data/test.csv", sep=",")

In [19]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 500],           # Number of trees in the forest
    'max_depth': [None, 5, 8, 12],             # Maximum depth (Control Overfitting)
    'min_samples_split': [2, 5, 10],           # Min samples required to split a node
    'min_samples_leaf': [1, 2, 4],             # Min samples required in a leaf
    'max_features': ['sqrt', 'log2']           # Number of features to consider at each split
}

rf_base = RandomForestClassifier()

grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    cv=5,
    n_jobs=-1,
    scoring="accuracy",
    verbose=True,
    refit=True)

grid_search.fit(X_train[:20000], y[:20000])

print("\n--- OPTIMIZATION RESULTS ---")
print(f"Best Hyperparameters: {grid_search.best_params_}")
print(f"Best CV Accuracy Score: {grid_search.best_score_ * 100:.2f}%")

Fitting 5 folds for each of 216 candidates, totalling 1080 fits

--- OPTIMIZATION RESULTS ---
Best Hyperparameters: {'max_depth': 8, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 200}
Best CV Accuracy Score: 87.70%


In [20]:
best_model = RandomForestClassifier(**grid_search.best_params_)
best_model.fit(X_train[:20000], y[:20000])


y_pred = best_model.predict(X_test)
y_pred = np.where(y_pred == "Presence", 1, 0)
y_pred_df = pd.DataFrame({
    "id" : df_test.id,
    "Heart Disease" : y_pred
})

y_pred_df.to_csv("data/kaggle_sub_rf_max_depth12_max_featuressqrt_minsamplesleaf4_minsamplessplit10_nestimators200.csv", index=False)
y_pred_df.head()

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- id
